# Iron March — 02: Análisis LLM
Extracción de entidades, topic modeling y análisis de contenido con LLM local.
Modelos: BERTopic (topic modeling) + qwen2.5:14b vía Ollama (NER y perfilado).

In [ ]:
import sys
import json
import re
from pathlib import Path
from collections import Counter
from itertools import combinations

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
import ollama
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer

from src.utils import RESULTS_DIR

plt.style.use('dark_background')
sns.set_palette('muted')

IM_RESULTS = RESULTS_DIR / 'ironmarch'
IM_RESULTS.mkdir(parents=True, exist_ok=True)

# Load cleaned parquets produced by 01_load_and_clean
posts = pd.read_parquet(IM_RESULTS / 'posts_clean.parquet')
users = pd.read_parquet(IM_RESULTS / 'users_clean.parquet')

print(f"posts:  {len(posts):,} filas  |  columnas: {list(posts.columns)}")
print(f"users:  {len(users):,} filas  |  columnas: {list(users.columns)}")

uid_to_name = dict(zip(users['userid'].astype(str), users['username'])) if 'username' in users.columns else {}


## Sección 1 — Topic Modeling (BERTopic)

BERTopic descubre los temas principales del foro de forma no supervisada.
Comparamos los topics automáticos con los subforos reales — si coinciden, el modelo es válido.

In [ ]:
# Preparar corpus de textos para topic modeling
# Filtramos posts cortos y usamos hasta 10.000 posts más largos para ser manejables
text_col = 'pagetext' if 'pagetext' in posts.columns else 'message'

corpus_df = posts[[text_col, 'userid'] + (['forumid'] if 'forumid' in posts.columns else [])].copy()
corpus_df = corpus_df.dropna(subset=[text_col])
corpus_df['text_clean'] = (
    corpus_df[text_col].astype(str)
    .str.replace(r'<[^>]+>', ' ', regex=True)
    .str.strip()
)
corpus_df = corpus_df[corpus_df['text_clean'].str.len() > 100]

# Muestra representativa: hasta 10.000 posts (los más largos)
if len(corpus_df) > 10000:
    corpus_df = corpus_df.nlargest(10000, 'text_clean')

documents = corpus_df['text_clean'].tolist()
print(f"Documentos para topic modeling: {len(documents):,}")


In [ ]:
# Ajustar BERTopic
vectorizer = CountVectorizer(stop_words='english', min_df=5, ngram_range=(1, 2))

topic_model = BERTopic(
    vectorizer_model=vectorizer,
    min_topic_size=15,
    nr_topics='auto',
    calculate_probabilities=False,
    verbose=True,
)

topics, _ = topic_model.fit_transform(documents)
corpus_df = corpus_df.copy()
corpus_df['topic'] = topics

n_topics = len(set(topics)) - (1 if -1 in topics else 0)
print(f"\nTopics encontrados: {n_topics}")
print(f"Posts sin topic (-1): {topics.count(-1):,}")

# Tabla resumen: topic_id, top words, count
topic_info = topic_model.get_topics()
valid_topics = {t: words for t, words in topic_info.items() if t != -1}
topic_sizes = {t: topics.count(t) for t in valid_topics}

rows = []
for t in sorted(topic_sizes, key=topic_sizes.get, reverse=True)[:20]:
    top_words = ', '.join([w for w, _ in valid_topics[t][:6]])
    rows.append({'topic_id': t, 'top_words': top_words, 'count': topic_sizes[t]})

topic_table = pd.DataFrame(rows)
print(topic_table.to_string(index=False))


In [ ]:
# Barchart: top 15 topics por tamaño
top_topic_ids = sorted(topic_sizes, key=topic_sizes.get, reverse=True)[:15]

labels = []
counts = []
for t in top_topic_ids:
    kws = [w for w, _ in valid_topics[t][:4]]
    labels.append(f'T{t}: {", ".join(kws)}')
    counts.append(topic_sizes[t])

fig = go.Figure(go.Bar(
    x=counts[::-1],
    y=labels[::-1],
    orientation='h',
    marker_color='#E94E4E',
))
fig.update_layout(
    title=f'Top 15 topics por tamaño — IronMarch ({n_topics} topics totales)',
    xaxis_title='Posts',
    template='plotly_dark',
    height=500, width=900,
    margin=dict(l=250, r=20, t=60, b=40),
)
fig.show()


In [ ]:
# Subplots de keywords por topic — top 12 topics
top_12 = sorted(topic_sizes, key=topic_sizes.get, reverse=True)[:12]
n_cols = 3
n_rows = (len(top_12) + n_cols - 1) // n_cols

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[f'Topic {t} ({topic_sizes[t]} posts)' for t in top_12],
    vertical_spacing=0.12, horizontal_spacing=0.08,
)
for idx, t in enumerate(top_12):
    row = idx // n_cols + 1
    col = idx % n_cols + 1
    words  = [w for w, _ in valid_topics[t][:8]]
    scores = [s for _, s in valid_topics[t][:8]]
    fig.add_trace(go.Bar(
        x=scores[::-1], y=words[::-1],
        orientation='h', marker_color='#E94E4E', showlegend=False,
    ), row=row, col=col)
    fig.update_xaxes(showticklabels=False, row=row, col=col)

fig.update_layout(
    title=f'Keywords por topic — IronMarch',
    template='plotly_dark',
    height=280 * n_rows, width=1100,
    margin=dict(l=10, r=10, t=80, b=10),
)
fig.show()


In [ ]:
# Usuarios más activos por topic — ¿quién lidera cada discurso?
if 'topic' in corpus_df.columns and 'userid' in corpus_df.columns:
    user_topic = (
        corpus_df[corpus_df['topic'] != -1]
        .groupby(['topic', 'userid'])
        .size()
        .reset_index(name='posts')
    )

    top_8 = sorted(topic_sizes, key=topic_sizes.get, reverse=True)[:8]
    rows_data = []
    for t in top_8:
        kws = [w for w, _ in topic_model.get_topic(t)[:3]]
        top_in_topic = (
            user_topic[user_topic['topic'] == t]
            .sort_values('posts', ascending=False)
            .head(5)
        )
        for _, row in top_in_topic.iterrows():
            uname = uid_to_name.get(str(row['userid']), str(row['userid']))
            rows_data.append({
                'topic': f'T{t}: {", ".join(kws)}',
                'usuario': uname,
                'posts_en_topic': int(row['posts']),
            })

    df_ut = pd.DataFrame(rows_data)
    fig = px.bar(
        df_ut, x='posts_en_topic', y='usuario', color='topic',
        facet_col='topic', facet_col_wrap=4,
        orientation='h',
        template='plotly_dark',
        title='Top 5 usuarios por topic (top 8 topics)',
        height=600, width=1100,
    )
    fig.update_yaxes(matches=None)
    fig.update_layout(showlegend=False)
    fig.show()


## Sección 2 — NER con LLM

Extracción de entidades nombradas (personas, organizaciones, lugares, ideologías, armas, eventos) desde los posts de los usuarios más activos, usando `qwen2.5:14b` vía Ollama.

Estrategia: top 50 usuarios por volumen de posts, hasta 50 posts por usuario.

In [ ]:
# Muestra: top 50 usuarios por cantidad de posts
text_col = 'pagetext' if 'pagetext' in posts.columns else 'message'

posts_text = posts.dropna(subset=[text_col]).copy()
posts_text = posts_text[posts_text[text_col].astype(str).str.len() > 100]

top_user_ids = posts_text.groupby('userid').size().nlargest(50).index.tolist()

sample_top = (
    posts_text[posts_text['userid'].isin(top_user_ids)]
    .groupby('userid')
    .apply(lambda g: g.nlargest(50, text_col) if len(g) > 50 else g)
    .reset_index(drop=True)
)

print(f"Usuarios en muestra: {sample_top['userid'].nunique()}")
print(f"Posts totales en muestra: {len(sample_top):,}")


In [ ]:
# Prompt del sistema y función de NER
SYSTEM_PROMPT = """You are an expert forensic analyst. Extract named entities from forum posts.
Return ONLY valid JSON object (not array): {"entities": [{"text": "...", "type": "PERSON|ORG|LOCATION|IDEOLOGY|WEAPON|EVENT", "confidence": 0.0-1.0}]}"""

def extract_entities(text: str, model: str = 'qwen2.5:14b') -> list[dict]:
    """Extracts named entities from a post using zero-shot NER via Ollama."""
    try:
        response = ollama.generate(
            model=model,
            system=SYSTEM_PROMPT,
            prompt=str(text)[:1500],
            format='json',
            options={'temperature': 0},
        )
        result = json.loads(response['response'])
        entities = result.get('entities', []) if isinstance(result, dict) else []
        return entities if isinstance(entities, list) else []
    except Exception:
        return []

print("Función NER lista.")
print("Verificando modelo disponible:")
import subprocess
subprocess.run(['ollama', 'list'], check=False)


In [ ]:
# Ejecutar NER con checkpoint cada 10 usuarios
NER_CACHE = IM_RESULTS / 'ner_results.parquet'
CHECKPOINT_EVERY = 10

if NER_CACHE.exists():
    ner_all = pd.read_parquet(NER_CACHE)
    print(f"NER cargado desde caché: {len(ner_all):,} entidades")
else:
    print(f"Ejecutando NER sobre {sample_top['userid'].nunique()} usuarios...")
    all_records = []
    user_list = sample_top['userid'].unique().tolist()

    for i, uid in enumerate(tqdm(user_list, desc='NER por usuario')):
        user_posts = sample_top[sample_top['userid'] == uid]
        for _, row in user_posts.iterrows():
            entities = extract_entities(str(row[text_col]))
            for ent in entities:
                txt = str(ent.get('text', '')).strip()
                if len(txt) < 2:
                    continue
                all_records.append({
                    'userid': uid,
                    'entity': txt,
                    'type': ent.get('type', 'UNKNOWN').upper(),
                    'confidence': float(ent.get('confidence', 0.5)),
                })

        # Checkpoint each batch of 10 users
        if (i + 1) % CHECKPOINT_EVERY == 0 and all_records:
            tmp = pd.DataFrame(all_records)
            tmp.to_parquet(NER_CACHE, index=False)
            print(f"  Checkpoint {i+1}/{len(user_list)} — {len(all_records):,} entidades")

    if all_records:
        ner_all = pd.DataFrame(all_records)
        ner_all = ner_all[ner_all['entity'].str.len() > 1]
        ner_all.to_parquet(NER_CACHE, index=False)
        print(f"NER guardado: {len(ner_all):,} entidades")
    else:
        print("⚠️  Sin resultados — verificar que qwen2.5:14b esté disponible.")
        print("    Ejecutar: ollama pull qwen2.5:14b")
        ner_all = pd.DataFrame(columns=['userid', 'entity', 'type', 'confidence'])
        ner_all.to_parquet(NER_CACHE, index=False)


In [ ]:
# Distribución de tipos de entidad
if len(ner_all) > 0:
    type_counts = ner_all['type'].value_counts()
    fig = go.Figure(go.Bar(
        x=type_counts.index.tolist(),
        y=type_counts.values.tolist(),
        marker_color=['#E94E4E', '#4E9EE9', '#E9A84E', '#7AE94E', '#C44EE9', '#FFD700'],
    ))
    fig.update_layout(
        title='Distribución de tipos de entidad — NER IronMarch',
        xaxis_title='Tipo', yaxis_title='Menciones',
        template='plotly_dark', height=380,
    )
    fig.show()

    # Top 10 entidades por tipo
    entity_types = [t for t in ['PERSON', 'ORG', 'ORGANIZATION', 'LOCATION', 'IDEOLOGY', 'WEAPON', 'EVENT']
                    if t in ner_all['type'].unique()]
    colors_map = {
        'PERSON': '#E94E4E', 'ORG': '#4E9EE9', 'ORGANIZATION': '#4E9EE9',
        'LOCATION': '#E9A84E', 'IDEOLOGY': '#7AE94E', 'WEAPON': '#FFD700', 'EVENT': '#C44EE9',
    }

    if entity_types:
        n_cols = min(len(entity_types), 3)
        n_rows = (len(entity_types) + n_cols - 1) // n_cols
        fig2 = make_subplots(
            rows=n_rows, cols=n_cols,
            subplot_titles=entity_types,
            horizontal_spacing=0.08,
            vertical_spacing=0.12,
        )
        for i, etype in enumerate(entity_types):
            row = i // n_cols + 1
            col = i % n_cols + 1
            subset = ner_all[ner_all['type'] == etype]
            top = subset['entity'].value_counts().head(10)
            fig2.add_trace(go.Bar(
                x=top.values[::-1],
                y=top.index[::-1].tolist(),
                orientation='h',
                marker_color=colors_map.get(etype, '#E94E4E'),
                showlegend=False,
            ), row=row, col=col)
        fig2.update_layout(
            title='Top 10 entidades por tipo — IronMarch',
            template='plotly_dark',
            height=380 * n_rows, width=380 * n_cols,
            margin=dict(l=140, r=20, t=60, b=40),
        )
        fig2.show()
else:
    print("Sin datos NER disponibles.")


In [ ]:
# Red de co-ocurrencia de entidades por usuario
if len(ner_all) > 0:
    # Filtrar tipos reveladores
    focus_types = [t for t in ['PERSON', 'IDEOLOGY', 'ORG', 'ORGANIZATION'] if t in ner_all['type'].unique()]
    ner_focus = ner_all[ner_all['type'].isin(focus_types)].copy()

    grouped = ner_focus.groupby('userid')['entity'].apply(list).reset_index()

    cooc = Counter()
    for _, row in grouped.iterrows():
        ents = list(set(row['entity']))
        for a, b in combinations(sorted(ents), 2):
            cooc[(a, b)] += 1

    cooc_filtered = {k: v for k, v in cooc.items() if v >= 2}

    if cooc_filtered:
        G_ner = nx.Graph()
        for (a, b), w in cooc_filtered.items():
            G_ner.add_edge(a, b, weight=w)

        top_nodes = sorted(G_ner.degree(), key=lambda x: x[1], reverse=True)[:40]
        top_set = {n for n, _ in top_nodes}
        G_viz = G_ner.subgraph(top_set).copy()

        pos = nx.spring_layout(G_viz, k=1.5, iterations=50, seed=42)

        edge_x, edge_y = [], []
        for u, v in G_viz.edges():
            x0, y0 = pos[u]; x1, y1 = pos[v]
            edge_x += [x0, x1, None]; edge_y += [y0, y1, None]

        edge_trace = go.Scatter(
            x=edge_x, y=edge_y, mode='lines',
            line=dict(width=0.8, color='rgba(150,150,150,0.3)'),
            hoverinfo='none',
        )
        node_deg = [G_viz.degree(n) for n in G_viz.nodes()]
        node_trace = go.Scatter(
            x=[pos[n][0] for n in G_viz.nodes()],
            y=[pos[n][1] for n in G_viz.nodes()],
            mode='markers+text',
            marker=dict(
                size=[d * 4 + 8 for d in node_deg],
                color=node_deg, colorscale='Plasma', showscale=True,
                colorbar=dict(title='Co-ocurrencias'),
            ),
            text=list(G_viz.nodes()),
            textposition='top center',
            textfont=dict(size=8, color='white'),
            hoverinfo='text',
        )
        fig = go.Figure(
            data=[edge_trace, node_trace],
            layout=go.Layout(
                title='Red de co-ocurrencia de entidades — IronMarch (PERSON, IDEOLOGY, ORG)',
                template='plotly_dark', showlegend=False,
                width=950, height=650,
                xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                margin=dict(l=20, r=20, t=60, b=20),
            )
        )
        fig.show()
        print(f"Red NER: {G_viz.number_of_nodes()} entidades, {G_viz.number_of_edges()} pares co-ocurrentes")
    else:
        print("Sin co-ocurrencias suficientes (mínimo 2). Ejecutar NER primero.")
else:
    print("Sin datos NER disponibles.")


## Sección 3 — Picos de actividad y temas

Conecta el análisis temporal con el análisis de contenido: para cada semana de actividad anómala (volumen > µ+2σ), identifica qué topics del BERTopic dominaron en esa ventana vs. el resto del período.

También incluye el índice de activación compuesto (volumen + longitud + mayúsculas + exclamaciones) y la evolución de narrativas clave en torno a los picos.

In [ ]:
# Detección de picos de actividad — semanas con volumen > µ + 2σ
if 'dateline' in posts.columns:
    valid_posts = posts.dropna(subset=['dateline']).copy()
    valid_posts = valid_posts[
        (valid_posts['dateline'].dt.year >= 2011) &
        (valid_posts['dateline'].dt.year <= 2018)
    ]

    weekly = valid_posts.groupby(valid_posts['dateline'].dt.to_period('W')).size()
    weekly_idx = [p.start_time for p in weekly.index]

    mu, sigma = weekly.mean(), weekly.std()
    threshold = mu + 2 * sigma
    spike_mask = weekly.values > threshold
    spike_periods = [p for p, is_spike in zip(weekly.index, spike_mask) if is_spike]

    print(f"Actividad semanal — µ={mu:.0f} posts/semana, σ={sigma:.0f}")
    print(f"Umbral de pico (µ+2σ): {threshold:.0f}")
    print(f"Semanas pico detectadas: {len(spike_periods)}")
    for p in spike_periods:
        print(f"  {p.start_time.strftime('%Y-%m-%d')}  →  {weekly[p]:,} posts")

    EVENTS = {
        '2011-07': 'Breivik', '2012-08': 'Wisconsin Sikh',
        '2015-06': 'Charleston', '2015-11': 'París',
        '2016-11': 'Trump', '2017-08': 'Charlottesville',
    }

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=weekly_idx, y=weekly.values,
        mode='lines', fill='tozeroy',
        line=dict(color='#E94E4E', width=1.5),
        fillcolor='rgba(233,78,78,0.15)',
        name='posts/semana',
    ))
    fig.add_hline(y=threshold, line_dash='dash', line_color='orange',
                  annotation_text=f'µ+2σ = {threshold:.0f}', annotation_position='top left')

    spike_x = [p.start_time for p in spike_periods]
    spike_y = [weekly[p] for p in spike_periods]
    fig.add_trace(go.Scatter(
        x=spike_x, y=spike_y, mode='markers',
        marker=dict(color='yellow', size=10, symbol='star'),
        name='pico',
    ))

    for ym, label in EVENTS.items():
        dt = pd.Timestamp(ym + '-01')
        fig.add_vline(x=dt, line_color='gold', line_dash='dot', opacity=0.6,
                      annotation_text=label, annotation_textangle=-90,
                      annotation_font_size=10)

    fig.update_layout(
        title='Actividad semanal IronMarch — picos estadísticos y eventos mundiales',
        xaxis_title='Semana', yaxis_title='Posts',
        template='plotly_dark', height=420,
    )
    fig.show()
else:
    print("Sin datos temporales.")
    spike_periods = []


In [ ]:
# Tópicos dominantes en cada semana pico
if not spike_periods:
    print("Ejecutar primero la celda de detección de picos.")
elif 'topic' not in corpus_df.columns:
    print("Ejecutar primero la sección BERTopic (topic en corpus_df no disponible).")
else:
    try:
        topic_src = corpus_df.copy()
        if 'dateline' not in topic_src.columns:
            topic_src = topic_src.join(posts[['dateline']], how='left')
        topic_src = topic_src.dropna(subset=['dateline'])

        # Labels de tópicos
        topic_labels = {}
        for _, row in topic_model.get_topic_info().iterrows():
            t = row['Topic']
            if t != -1:
                kws = [w for w, _ in topic_model.get_topic(t)[:4]]
                topic_labels[t] = ' '.join(kws)
            else:
                topic_labels[t] = 'sin_topic'

        results = []
        for sp in spike_periods:
            tz = topic_src['dateline'].dt.tz
            start = pd.Timestamp(sp.start_time, tz=tz) - pd.Timedelta(weeks=1)
            end   = pd.Timestamp(sp.start_time, tz=tz) + pd.Timedelta(weeks=2)
            mask  = (topic_src['dateline'] >= start) & (topic_src['dateline'] <= end)
            window = topic_src[mask]
            if len(window) == 0:
                continue
            top_topics = window[window['topic'] != -1]['topic'].value_counts().head(5)
            for t, cnt in top_topics.items():
                results.append({
                    'pico': sp.start_time.strftime('%Y-%m-%d'),
                    'label': topic_labels.get(t, str(t))[:35],
                    'posts': cnt,
                })

        if results:
            res_df = pd.DataFrame(results)
            spikes_list = res_df['pico'].unique().tolist()
            n = len(spikes_list)
            cols = min(n, 3)
            rows = (n + cols - 1) // cols
            fig = make_subplots(rows=rows, cols=cols,
                                subplot_titles=[f'Pico {p}' for p in spikes_list])
            for k, spike_date in enumerate(spikes_list):
                r, c = divmod(k, cols)
                sub = res_df[res_df['pico'] == spike_date].sort_values('posts', ascending=True)
                fig.add_trace(go.Bar(
                    x=sub['posts'], y=sub['label'],
                    orientation='h', marker_color='#E94E4E', showlegend=False,
                ), row=r + 1, col=c + 1)
            fig.update_layout(
                title='Tópicos dominantes en semanas pico (ventana ±1 semana)',
                template='plotly_dark', height=350 * rows,
            )
            fig.show()
        else:
            print("Sin datos de tópicos en ventanas pico.")
    except Exception as e:
        print(f"Error: {e}")
        import traceback; traceback.print_exc()


In [ ]:
# Índice de activación del foro — proxy de intensidad emocional
# Componentes: volumen + longitud media + ratio de mayúsculas + ratio de exclamaciones
if 'dateline' in posts.columns:
    text_col_act = 'pagetext' if 'pagetext' in posts.columns else 'message'
    p = posts.dropna(subset=['dateline', text_col_act]).copy()
    p = p[(p['dateline'].dt.year >= 2011) & (p['dateline'].dt.year <= 2018)]
    p['text_str']   = p[text_col_act].astype(str)
    p['nchars']     = p['text_str'].str.len()
    p['cap_ratio']  = p['text_str'].apply(lambda t: sum(1 for c in t if c.isupper()) / max(len(t), 1))
    p['excl_ratio'] = p['text_str'].str.count('!') / p['text_str'].str.len().clip(lower=1)
    p['week']       = p['dateline'].dt.to_period('W')

    weekly_act = p.groupby('week').agg(
        n_posts    = ('nchars', 'count'),
        mean_len   = ('nchars', 'mean'),
        cap_ratio  = ('cap_ratio', 'mean'),
        excl_ratio = ('excl_ratio', 'mean'),
    )

    def zscore(s):
        return (s - s.mean()) / max(s.std(), 1e-9)

    weekly_act['activation'] = (
        zscore(weekly_act['n_posts']) +
        zscore(weekly_act['mean_len']) +
        zscore(weekly_act['cap_ratio']) +
        zscore(weekly_act['excl_ratio'])
    ) / 4

    wx = [period.start_time for period in weekly_act.index]

    EVENTS = {
        '2011-07': 'Breivik', '2012-08': 'Wisconsin Sikh',
        '2015-06': 'Charleston', '2015-11': 'París',
        '2016-11': 'Trump', '2017-08': 'Charlottesville',
    }

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=wx, y=weekly_act['activation'],
        mode='lines', fill='tozeroy',
        line=dict(color='#FF6B35', width=1.5),
        fillcolor='rgba(255,107,53,0.15)',
        name='activación',
    ))
    fig.add_hline(y=0, line_color='gray', line_dash='dot')

    for ym, label in EVENTS.items():
        dt = pd.Timestamp(ym + '-01')
        fig.add_vline(x=dt, line_color='gold', line_dash='dot', opacity=0.6,
                      annotation_text=label, annotation_textangle=-90,
                      annotation_font_size=10)

    top5 = weekly_act['activation'].nlargest(5)
    fig.add_trace(go.Scatter(
        x=[period.start_time for period in top5.index],
        y=top5.values,
        mode='markers', marker=dict(color='yellow', size=10, symbol='star'),
        name='top activación',
    ))

    fig.update_layout(
        title='Índice de activación semanal IronMarch (z-score compuesto)',
        xaxis_title='Semana', yaxis_title='Activación (z-score)',
        template='plotly_dark', height=420,
    )
    fig.show()

    print("Top 5 semanas de mayor activación:")
    for period, val in top5.items():
        row = weekly_act.loc[period]
        print(f"  {period.start_time.strftime('%Y-%m-%d')}  activación={val:.2f}  "
              f"posts={int(row['n_posts'])}  len_media={row['mean_len']:.0f}")
else:
    print("Sin datos temporales.")


In [ ]:
# Serie temporal de narrativas clave — normalizada por posts/semana
if 'dateline' in posts.columns:
    text_col_ts = 'pagetext' if 'pagetext' in posts.columns else 'message'
    p = posts.dropna(subset=['dateline', text_col_ts]).copy()
    p = p[(p['dateline'].dt.year >= 2013) & (p['dateline'].dt.year <= 2017)]
    p['text_lower'] = p[text_col_ts].astype(str).str.lower().apply(
        lambda t: re.sub(r'<[^>]+>', ' ', t)
    )
    p['week'] = p['dateline'].dt.to_period('W')

    KW_GROUPS = {
        'Islam / jihad':          ['islam', 'muslim', 'jihad', 'isis', 'allah'],
        'Refugees / migration':   ['refugee', 'migrant', 'immigration', 'invasi'],
        'France / Paris':         ['paris', 'france', 'french'],
        'Race / supremacy':       ['white', 'race', 'aryan', 'negro', 'black'],
        'Jews':                   ['jew', 'jewish', 'zion'],
    }
    COLORS = ['#E94E4E', '#FF9500', '#FFD700', '#4EC9FF', '#A78BFA']

    def weekly_kw_rate(df, keywords):
        pattern = '|'.join([r'\b' + kw + r'\w*\b' for kw in keywords])
        df = df.copy()
        df['hits'] = df['text_lower'].str.count(pattern)
        weekly = df.groupby('week').agg(hits=('hits', 'sum'), n=('hits', 'count'))
        weekly['rate'] = weekly['hits'] / weekly['n'].clip(lower=1) * 100
        return weekly['rate']

    fig = go.Figure()
    for (group, kws), color in zip(KW_GROUPS.items(), COLORS):
        rate = weekly_kw_rate(p, kws)
        wx = [period.start_time for period in rate.index]
        fig.add_trace(go.Scatter(
            x=wx, y=rate.values, mode='lines', name=group,
            line=dict(color=color, width=1.8),
            hovertemplate=f'<b>{group}</b><br>%{{x|%Y-%m-%d}}<br>%{{y:.1f}} menciones/100 posts<extra></extra>',
        ))

    MARKERS = {
        '2015-01-07': 'Charlie Hebdo',
        '2015-11-13': 'Atentados París',
        '2016-07-14': 'Niza',
        '2016-11-08': 'Trump elegido',
        '2017-08-12': 'Charlottesville',
    }
    for date_str, label in MARKERS.items():
        dt = pd.Timestamp(date_str)
        fig.add_vline(x=dt, line_color='white', line_dash='dot',
                      line_width=1, opacity=0.5,
                      annotation_text=label, annotation_textangle=-90,
                      annotation_font_size=10, annotation_font_color='white')

    fig.update_layout(
        title='Evolución temporal de narrativas clave — IronMarch 2013–2017<br>'
              '<sup>Frecuencia normalizada por posts/semana</sup>',
        xaxis_title='Semana',
        yaxis_title='Menciones por 100 posts',
        template='plotly_dark',
        height=500,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        hovermode='x unified',
    )
    fig.show()

    # Ratio de narrativas en ventana del pico de París vs. baseline mayo 2015
    print("\n── Ratio vs. baseline en ventana de París (nov 2015) ──")
    base_s  = pd.Timestamp('2015-04-30', tz='UTC') if p['dateline'].dt.tz else pd.Timestamp('2015-04-30')
    base_e  = pd.Timestamp('2015-05-27', tz='UTC') if p['dateline'].dt.tz else pd.Timestamp('2015-05-27')
    spike_s = pd.Timestamp('2015-10-30', tz='UTC') if p['dateline'].dt.tz else pd.Timestamp('2015-10-30')
    spike_e = pd.Timestamp('2015-11-27', tz='UTC') if p['dateline'].dt.tz else pd.Timestamp('2015-11-27')

    p_base  = p[(p['dateline'] >= base_s)  & (p['dateline'] <= base_e)]
    p_spike = p[(p['dateline'] >= spike_s) & (p['dateline'] <= spike_e)]

    print(f"{'Narrativa':<32} {'Base':>8} {'Pico':>8} {'ratio':>7}")
    print("-" * 57)
    for group, kws in KW_GROUPS.items():
        pattern = '|'.join([r'\b' + kw + r'\w*\b' for kw in kws])
        def rate(df):
            hits = df['text_lower'].str.count(pattern).sum()
            return hits / max(len(df), 1) * 100
        rb, rs = rate(p_base), rate(p_spike)
        arrow = ' ⬆' if rs / max(rb, 0.001) > 1.5 else (' ⬇' if rs / max(rb, 0.001) < 0.7 else '')
        print(f"  {group:<30} {rb:>7.1f} {rs:>8.1f} {rs / max(rb, 0.001):>6.2f}x{arrow}")
else:
    print("Sin datos temporales.")


## Sección 4 — Guardar resultados

Persistir los resultados del topic modeling y NER en Parquet para consumo posterior.

In [ ]:
# Guardar topic model results
if 'topic' in corpus_df.columns:
    topic_out = corpus_df[['userid', 'topic'] + (
        ['forumid'] if 'forumid' in corpus_df.columns else []
    )].copy()
    topic_out['topic_label'] = topic_out['topic'].map(
        lambda t: ', '.join([w for w, _ in topic_model.get_topic(t)[:4]]) if t != -1 else 'outlier'
    )
    topic_path = IM_RESULTS / 'topic_model_results.parquet'
    topic_out.to_parquet(topic_path, index=False)
    print(f"Topic model results guardado: {topic_path}")
    print(f"  {len(topic_out):,} posts con topic asignado")
else:
    print("Sin resultados de topic modeling para guardar.")

# NER results ya se guardan con checkpoint — confirmar
ner_path = IM_RESULTS / 'ner_results.parquet'
if ner_path.exists():
    ner_check = pd.read_parquet(ner_path)
    print(f"NER results confirmado: {ner_path}")
    print(f"  {len(ner_check):,} entidades | {ner_check['type'].value_counts().to_dict()}")
else:
    print("NER results no disponible — ejecutar Sección 2.")
